# Pothole YOLOX-Nano · Colab / Telefon modeli
Run on a free GPU. No model has been trained by this notebook's author. Expect a multi-hour resumable run, a 13.3 GB download and sufficient disk space. Start from scratch; no third-party pretrained weights. The export stops if validation precision 0.80 is not achieved.

In [ ]:
import os, subprocess, json
from pathlib import Path
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'Runtime → Change runtime type → GPU'
from google.colab import drive
drive.mount('/content/drive')  # Interactive Colab authorization; never save tokens.
if not Path('/content/pothole').exists():
    subprocess.run(['git','clone','--branch','ondevice-ml','https://github.com/sabdur4hmonov/potholedetector.git','/content/pothole'], check=True)
os.chdir('/content/pothole')
print('App source:', subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())
pins=json.loads(Path('ml/pins.json').read_text())
work=Path('/content/drive/MyDrive/pothole-yolox'); work.mkdir(parents=True,exist_ok=True)
# Preserve the first measured dataset pin across runtime restarts.
if (work/'pins.json').exists():
    saved=json.loads((work/'pins.json').read_text())
    assert saved['yolox_commit']==pins['yolox_commit']
    pins['rdd2022_sha256']=saved['rdd2022_sha256']
    Path('ml/pins.json').write_text(json.dumps(pins,indent=2))

In [ ]:
import sys
subprocess.run([sys.executable,'-m','pip','install','torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','ml/requirements-colab.txt','setuptools==75.8.0','wheel==0.45.1','packaging==24.2'],check=True)
if not Path('/content/YOLOX').exists():
    subprocess.run(['git','clone','https://github.com/Megvii-BaseDetection/YOLOX.git','/content/YOLOX'],check=True)
subprocess.run(['git','-C','/content/YOLOX','checkout','--detach',pins['yolox_commit']],check=True)
assert subprocess.check_output(['git','-C','/content/YOLOX','rev-parse','HEAD']).decode().strip()==pins['yolox_commit']
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--no-build-isolation','-e','/content/YOLOX'],check=True)

Optional owner data: upload a manual dataset ZIP to `/content/owner_export.zip`. Wrong/No frames are negatives. Yes frames require human COCO boxes in labels.json before detection training; unboxed positives are retained in owner_positive_review.json and never mislabeled as negatives. This does not use model boxes as ground truth. Only full frames are used.

In [ ]:
import shutil
args=[sys.executable,'ml/prepare_rdd2022.py','--archive','/content/RDD2022.zip','--output','/content/pothole/ml/data']
if Path('/content/owner_export.zip').exists(): args += ['--extra','/content/owner_export.zip']
subprocess.run(args,check=True)
for name in ('pins.json','dataset_stats.json'):
    shutil.copyfile(Path('ml')/name,work/name)
print(Path('ml/dataset_stats.json').read_text())

In [ ]:
# Resume the latest checkpoint automatically; adjust epochs/batch to the GPU/session.
os.environ['POTHOLE_DATA']='/content/pothole/ml/data'
os.environ['POTHOLE_CHECKPOINTS']=str(work)
os.environ['POTHOLE_EPOCHS']='40'
checkpoint=work/'pothole_nano/latest_ckpt.pth'
command=[sys.executable,'/content/YOLOX/tools/train.py','-f','/content/pothole/ml/yolox_pothole_exp.py','-d','1','-b','16','--fp16','-o']
if checkpoint.exists(): command += ['--resume','-c',str(checkpoint)]
subprocess.run(command,check=True)

In [ ]:
best=work/'pothole_nano/best_ckpt.pth'
assert best.exists(), 'Training did not produce a best checkpoint'
subprocess.run([sys.executable,'ml/evaluate_export.py','--data',os.environ['POTHOLE_DATA'],'--checkpoint',str(best),'--output','/content/model_export'],check=True)
for name in ('pothole_yolox_nano.onnx','pothole_model.json'):
    shutil.copyfile(Path('/content/model_export')/name,work/name)
print('Test metrics and per-country recall are above; no Uzbekistan field accuracy claim.')

In [ ]:
from google.colab import files
files.download('/content/model_export/pothole_yolox_nano.onnx')
files.download('/content/model_export/pothole_model.json')
# Also retain measured dataset pin/stats for review.
files.download(str(work/'pins.json'))
files.download(str(work/'dataset_stats.json'))

Put the two model files in `android-app/android/app/src/main/assets/ml/`, then tell Codex/Claude to verify them and rebuild. Do not commit weights or datasets. Test while parked before field validation. Default warning timing is approximate and not navigation.